## Comparing Encoder-Decoder, Encoder-Only and Decoder-Only models

In [3]:
from transformers import T5Tokenizer, T5ForConditionalGeneration  # Encoder-Decoder
from transformers import BertTokenizer, BertForMaskedLM           # Encoder-Only
from transformers import GPT2Tokenizer, GPT2LMHeadModel           # Decoder-Only

# Sample input text
input_text = "Transformers are powerful tools for natural language processing."

In [4]:
# === Encoder-Decoder Model (e.g., T5) ===
print("=== Encoder-Decoder Model (T5) ===")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Encode input for T5
t5_input = t5_tokenizer.encode("summarize: " + input_text, return_tensors="pt", max_length=512, truncation=True)
t5_output = t5_model.generate(t5_input, max_length=50)

# Decode and print output
t5_decoded_output = t5_tokenizer.decode(t5_output[0], skip_special_tokens=True)
print(f"T5 Output: {t5_decoded_output}\n")



=== Encoder-Decoder Model (T5) ===


model.safetensors:  69%|######9   | 168M/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5 Output: Transformers are powerful tools for natural language processing.



In [5]:
# === Encoder-Only Model (e.g., BERT) ===
print("=== Encoder-Only Model (BERT) ===")
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Mask a token in the input for BERT
masked_input = input_text.replace("tools", "[MASK]")
bert_input = bert_tokenizer(masked_input, return_tensors="pt", max_length=512, truncation=True)

# Predict the masked token
bert_output = bert_model(**bert_input)
predicted_token_id = bert_output.logits[0, bert_input['input_ids'][0] == bert_tokenizer.mask_token_id].argmax().item()
predicted_token = bert_tokenizer.decode(predicted_token_id)

print(f"Original Input: {input_text}")
print(f"Masked Input: {masked_input}")
print(f"Predicted Token: {predicted_token}\n")

=== Encoder-Only Model (BERT) ===


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

Original Input: Transformers are powerful tools for natural language processing.
Masked Input: Transformers are powerful [MASK] for natural language processing.
Predicted Token: tools



In [6]:
# === Decoder-Only Model (e.g., GPT-2) ===
print("=== Decoder-Only Model (GPT-2) ===")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")

# Encode input for GPT-2
gpt2_input = gpt2_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
gpt2_output = gpt2_model.generate(gpt2_input, max_length=50, num_return_sequences=1)

# Decode and print output
gpt2_decoded_output = gpt2_tokenizer.decode(gpt2_output[0], skip_special_tokens=True)
print(f"GPT-2 Output: {gpt2_decoded_output}\n")

=== Decoder-Only Model (GPT-2) ===


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


GPT-2 Output: Transformers are powerful tools for natural language processing.

The following is a list of the most common languages used by the Python interpreter.

Python

Python is a programming language that is used to write and execute code. It is a



## Beam Search and Be Creative

In [7]:
import tensorflow as tf
from transformers import GPT2Tokenizer, TFGPT2LMHeadModel

# Load pre-trained GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = TFGPT2LMHeadModel.from_pretrained("gpt2")

# Sample input text
input_text = "Transformers are powerful tools"
input_ids = tokenizer.encode(input_text, return_tensors="tf")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# === Beam Search ===
print("=== Beam Search ===")
beam_output = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,  # Number of beams to explore
    early_stopping=True,  # Stop when the best beam finishes
    no_repeat_ngram_size=2  # Prevent repeating n-grams
)

# Decode and print beam search output
beam_decoded_output = tokenizer.decode(beam_output[0], skip_special_tokens=True)
print(f"Beam Search Output: {beam_decoded_output}\n")

In [ ]:
# === Random Sampling with tf.random.categorical ===
print("=== Random Sampling with tf.random.categorical ===")

# Generate logits using the model
outputs = model(input_ids)
logits = outputs.logits[:, -1, :]  # Get logits for the last token

# Apply softmax to convert logits to probabilities
probs = tf.nn.softmax(logits, axis=-1)

# Sample from the categorical distribution
sampled_token_id = tf.random.categorical(probs, num_samples=1)[0][0].numpy()

# Decode the sampled token
sampled_token = tokenizer.decode(sampled_token_id)
print(f"Sampled Token: {sampled_token}")

# Generate a full sequence using random sampling
random_sample_output = model.generate(
    input_ids,
    max_length=50,
    do_sample=True,  # Enable random sampling
    top_k=50,  # Limit sampling to top-k tokens
    temperature=0.7  # Control randomness (lower = more deterministic)
)

# Decode and print random sampling output
random_sample_decoded_output = tokenizer.decode(random_sample_output[0], skip_special_tokens=True)
print(f"Random Sampling Output: {random_sample_decoded_output}")

## Masked token prediction and Next token prediction

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM  # For Mask Token Prediction
from transformers import GPT2Tokenizer, GPT2LMHeadModel  # For Next Token Prediction
import torch

# === Mask Token Prediction (BERT) ===
print("=== Mask Token Prediction (BERT) ===")
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Input text with a masked token
masked_input_text = "Transformers are powerful [MASK] for natural language processing."
bert_input_ids = bert_tokenizer.encode(masked_input_text, return_tensors="pt")

# Get predictions from BERT
with torch.no_grad():
    bert_outputs = bert_model(bert_input_ids)
    predictions = bert_outputs.logits

# Find the position of the [MASK] token
mask_token_index = torch.where(bert_input_ids == bert_tokenizer.mask_token_id)[1]
mask_token_logits = predictions[0, mask_token_index, :]

# Get the top predicted token
top_predicted_token_id = torch.argmax(mask_token_logits, dim=-1).item()
top_predicted_token = bert_tokenizer.decode(top_predicted_token_id)

print(f"Original Input: {masked_input_text}")
print(f"Predicted Token for [MASK]: {top_predicted_token}\n")

In [ ]:
# === Next Token Prediction (GPT-2) ===
print("=== Next Token Prediction (GPT-2) ===")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")

# Input text for next token prediction
input_text = "Transformers are powerful tools"
gpt2_input_ids = gpt2_tokenizer.encode(input_text, return_tensors="pt")

# Get predictions from GPT-2
with torch.no_grad():
    gpt2_outputs = gpt2_model(gpt2_input_ids)
    next_token_logits = gpt2_outputs.logits[:, -1, :]  # Logits for the last token

# Get the top predicted token
top_next_token_id = torch.argmax(next_token_logits, dim=-1).item()
top_next_token = gpt2_tokenizer.decode(top_next_token_id)

print(f"Input Text: {input_text}")
print(f"Predicted Next Token: {top_next_token}")

## Traditional Fine Tunning

In [ ]:
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

# === Step 1: Load Dataset ===
print("Loading dataset...")
dataset = load_dataset("imdb")  # IMDB dataset for sentiment analysis
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))  # Use a subset for demonstration
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

# === Step 2: Tokenize Inputs ===
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing dataset...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns and set format for PyTorch
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"]).with_format("torch")
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"]).with_format("torch")

# === Step 3: Prepare Model ===
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# === Step 4: Define Training Arguments ===
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# === Step 5: Train the Model ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

# === Step 6: Evaluate the Model ===
print("Evaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# === Step 7: Save the Fine-Tuned Model ===
print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_distilbert")
tokenizer.save_pretrained("./fine_tuned_distilbert")